In [ ]:
import json
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from scipy.optimize import curve_fit
from huggingface_hub import snapshot_download
from matplotlib.ticker import FuncFormatter, NullFormatter

import plot_utils as pu
from plot_utils import create_figure, save_figure, human_readable_format

def truncate_label(label: str, max_len: int = 10) -> str:
    """Shorten labels while preserving a terminal ellipsis when truncated."""
    if len(label) <= max_len:
        return label
    if max_len <= 3:
        return label[:max_len]
    return f"{label[: max_len - 6]}...{label[-3:]}"


def unique_truncated_labels(labels, max_len: int = 10):
    """Ensure truncated labels remain unique by appending numeric suffixes when needed."""
    seen = set()
    duplicate_counts = {}
    unique = []
    for label in labels:
        base = truncate_label(label, max_len=max_len)
        duplicate_counts.setdefault(base, 0)
        candidate = base
        while candidate in seen:
            duplicate_counts[base] += 1
            suffix = f"_{duplicate_counts[base] + 1}"
            keep_len = max_len - len(suffix)
            if keep_len <= 0:
                candidate = suffix[-max_len:]
            else:
                candidate = f"{base[:keep_len]}{suffix}"
        seen.add(candidate)
        unique.append(candidate)
    return unique

# Fit Piecewise Power Law distribution with up to three tiers
def zipf_func(x, alpha, C):
    return C / (x ** alpha)

def enumerate_breakpoints(n_points: int, segments: int, min_points: int = 2):
    """Yield valid breakpoint combinations ensuring each segment has >= min_points."""

    if segments <= 1:
        yield ()
        return

    def helper(prev_index: int, segments_remaining: int):
        if segments_remaining == 1:
            yield ()
            return
        min_break = prev_index + min_points
        max_break = n_points - min_points * (segments_remaining - 1)
        for breakpoint in range(min_break, max_break + 1):
            for rest in helper(breakpoint, segments_remaining - 1):
                yield (breakpoint,) + rest

    yield from helper(0, segments)

def fit_piecewise_zipf(rank_values, token_values, min_points: int = 2, max_segments: int = 3):
    """Search for the best multi-tier Zipf fit measured by log-space R²."""

    n_points = len(rank_values)
    log_tokens = np.log10(token_values)
    ss_tot_log = np.sum((log_tokens - np.mean(log_tokens)) ** 2)

    best_fit = {
        "segments": 1,
        "breakpoints": [],
        "params": [],
        "pred": None,
        "tier_ids": np.zeros(n_points, dtype=int),
        "r2": -np.inf,
    }

    def evaluate_segments(segments: int):
        for breakpoints in enumerate_breakpoints(n_points, segments, min_points):
            start = 0
            preds = []
            params = []
            valid = True
            for end in list(breakpoints) + [n_points]:
                segment_rank = rank_values[start:end]
                segment_tokens = token_values[start:end]
                if len(segment_rank) < 2:
                    valid = False
                    break
                try:
                    popt, _ = curve_fit(zipf_func, segment_rank, segment_tokens, p0=[1.0, segment_tokens[0]])
                except Exception:
                    valid = False
                    break
                params.append(popt)
                preds.append(zipf_func(segment_rank, *popt))
                start = end
            if not valid:
                continue
            pred_all = np.concatenate(preds)
            if not np.isfinite(pred_all).all():
                continue
            if ss_tot_log == 0:
                r2_piece = 1.0
            else:
                log_pred = np.log10(pred_all)
                ss_res = np.sum((log_tokens - log_pred) ** 2)
                r2_piece = 1 - (ss_res / ss_tot_log)
            if r2_piece > best_fit["r2"]:
                tier_ids = np.zeros(n_points, dtype=int)
                edges = [0] + list(breakpoints) + [n_points]
                for tier_idx in range(len(edges) - 1):
                    tier_ids[edges[tier_idx] : edges[tier_idx + 1]] = tier_idx
                best_fit.update(
                    {
                        "segments": segments,
                        "breakpoints": list(breakpoints),
                        "params": params,
                        "pred": pred_all,
                        "tier_ids": tier_ids,
                        "r2": r2_piece,
                    }
                )

    max_segments_allowed = min(max_segments, len(top_models_bar) // min_points)
    for segment_count in range(2, max_segments_allowed + 1):
        evaluate_segments(segment_count)

    if best_fit["pred"] is None:
        popt, _ = curve_fit(zipf_func, rank_values, token_values, p0=[1.0, token_values[0]])
        best_fit.update(
            {
                "segments": 1,
                "breakpoints": [],
                "params": [popt],
                "pred": zipf_func(rank_values, *popt),
                "tier_ids": np.zeros(len(rank_values), dtype=int),
                "r2": 0.0,
            }
        )
    return best_fit

In [ ]:
local_dir = snapshot_download(
    repo_id="eth-easl/swissai-serving-trace",
    repo_type="dataset",
    allow_patterns=["trace.jsonl", "qwen3-32b-buckets.jsonl"],
)
with (Path(local_dir) / "trace.jsonl").open("r", encoding="utf-8") as handle:
    data = [json.loads(line) for line in handle]
df = pd.DataFrame(data)
print(f"Loaded {len(df)} records from trace.jsonl")
df = df[(df["status"] == 'DEFAULT') & (df["created_at"] != '\\N') & (df["finished_at"] != '\\N')]
df['created_at'] = pd.to_datetime(df['created_at'], format='ISO8601', utc=True)
df['finished_at'] = pd.to_datetime(df['finished_at'], format='ISO8601', utc=True)
df['latency_s'] = (df['finished_at'] - df['created_at']).dt.total_seconds()
df['total_tokens'] = df['reported_token_input'] + df['reported_token_output']
df.head()

Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 836.60it/s]


Loaded 16329237 records from trace.jsonl


reported_token_input  reported_token_output  \
3                    111                    184   
5                    144                    269   
8                     96                    132   
11                    22                   1628   
13                    24                    665   

                                                   id   status  \
3   479869c90e8f0280:ff9095fbbcb6c1e38bc893148132e...  DEFAULT   
5   479a50953e392b63:d61c9eaeb73af7963ab9b3332198b...  DEFAULT   
8   479e8222a2eb5327:b0510c7cf0a062f105833de305433...  DEFAULT   
11  47a0ca142bee320b:2d5ee005b11ea38f10867ee5069c3...  DEFAULT   
13  47a16ec6a8e78aed:80674dc9f43c059a8266363c4c508...  DEFAULT   

                         created_at                      finished_at  \
3  2025-08-21 19:19:22.469000+00:00 2025-08-21 19:19:27.249000+00:00   
5  2025-08-21 16:22:04.494000+00:00 2025-08-21 16:22:11.766000+00:00   
8  2025-08-21 19:21:35.523000+00:00 2025-08-21 19:21:38.797000+00:00   
11 2025-08-21 21:32:39.305000+00:00 2025-08-21 21:34:18.119000+00:00   
13 2025-08-21 21:24:00.788000+00:00 2025-08-21 21:24:33.661000+00:00   

             model                                   model_parameters  \
3   Qwen/Qwen3-32B  {'temperature': 0.7, 'max_tokens': 'null', 'to...   
5   Qwen/Qwen3-32B  {'temperature': 0.7, 'max_tokens': 'null', 'to...   
8   Qwen/Qwen3-32B  {'temperature': 0.7, 'max_tokens': 'null', 'to...   
11  Qwen/Qwen3-32B  {'temperature': 0.7, 'max_tokens': 'null', 'to...   
13  Qwen/Qwen3-32B  {'temperature': 0.7, 'max_tokens': 'null', 'to...   

    latency_s  total_tokens  
3       4.780           295  
5       7.272           413  
8       3.274           228  
11     98.814          1650  
13     32.873           689

In [9]:
# exclude >1000s latency calls to focus on main distribution and avoid long tail skewing
short_call_df = df[df['latency_s'] <= 1000]
qwen3_thinking_df = df[df['model'] == 'Qwen/Qwen3-Next-80B-A3B-Thinking']
qwen3_short_df = qwen3_thinking_df[qwen3_thinking_df['latency_s'] <= 1000]

p95_latency_all = df['latency_s'].quantile(0.95)
p99_latency_all = df['latency_s'].quantile(0.99)
mean_latency_all = df['latency_s'].mean()
qwen3_latency_mean = qwen3_thinking_df['latency_s'].mean()
qwen3_latency_p95 = qwen3_thinking_df['latency_s'].quantile(0.95)
qwen3_latency_p99 = qwen3_thinking_df['latency_s'].quantile(0.99)

# Derive latency per generated token (s/token) for rows with positive token counts
df['latency_per_token'] = np.nan
positive_token_mask = df['total_tokens'] > 0
df.loc[positive_token_mask, 'latency_per_token'] = (
    df.loc[positive_token_mask, 'latency_s'] / df.loc[positive_token_mask, 'total_tokens']
 )
latency_per_token_df = df[positive_token_mask & df['latency_per_token'].notna()].copy()

latency_per_token_short = latency_per_token_df[latency_per_token_df['latency_per_token'] <= 0.5]
latency_per_token_series = latency_per_token_short['latency_per_token']

if latency_per_token_series.empty:
    raise ValueError('No valid latency-per-token samples available for plotting.')

latency_per_token_mean = latency_per_token_series.mean()
latency_per_token_p95 = latency_per_token_series.quantile(0.95)
latency_per_token_p99 = latency_per_token_series.quantile(0.99)

# Split latency-per-token samples by model to compare columns
qwen3_model_name = 'Qwen/Qwen3-Next-80B-A3B-Thinking'
qwen3_latency_per_token_series = latency_per_token_short.loc[
    latency_per_token_short['model'] == qwen3_model_name,
    'latency_per_token',
]
other_latency_per_token_series = latency_per_token_short.loc[
    latency_per_token_short['model'] != qwen3_model_name,
    'latency_per_token',
]
qwen3_latency_per_token_mean = (
    qwen3_latency_per_token_series.mean()
    if not qwen3_latency_per_token_series.empty
    else None
)
qwen3_latency_per_token_p95 = (
    qwen3_latency_per_token_series.quantile(0.95)
    if not qwen3_latency_per_token_series.empty
    else None
)
qwen3_latency_per_token_p99 = (
    qwen3_latency_per_token_series.quantile(0.99)
    if not qwen3_latency_per_token_series.empty
    else None
)

metric_styles = {
    'mean': {'color': '#1A3D6D', 'linestyle': ':', 'linewidth': 1.8},
    'p95': {'color': '#DD8452', 'linestyle': '-.', 'linewidth': 1.8},
    'p99': {'color': '#937860', 'linestyle': '-', 'linewidth': 1.8},
}

def add_metric_lines(ax, metrics):
    for metric_name, value in metrics.items():
        if value is None or not np.isfinite(value):
            continue
        style = metric_styles[metric_name]
        ax.axvline(value, **style)

# Figure 1: End-to-end latency distributions (All vs Thinking)
fig_e2e, axes_e2e = create_figure(
    "double_column",
    nrows=1,
    ncols=2,
    figsize=(9, 3.5),
)
axes_e2e = np.atleast_1d(axes_e2e).ravel()
ax_e2e_all, ax_e2e_qwen = axes_e2e
ax_e2e_all.grid(linestyle=":", linewidth=2)
ax_e2e_qwen.grid(linestyle=":", linewidth=2)

bins_latency = np.linspace(0, 1000, 100)
ax_e2e_all.hist(
    short_call_df['latency_s'],
    bins=bins_latency,
    color='#4C72B0',
    edgecolor='white',
    alpha=0.75,
    label=None,
 )
add_metric_lines(
    ax_e2e_all,
    {
        'mean': mean_latency_all,
        'p95': p95_latency_all,
        'p99': p99_latency_all,
    },
 )
ax_e2e_all.set_title('E2E (All)')
ax_e2e_all.set_yscale('log')
ax_e2e_qwen.hist(
    qwen3_short_df['latency_s'],
    bins=bins_latency,
    color='#DD8452',
    edgecolor='white',
    alpha=0.8,
    label=None,
 )
add_metric_lines(
    ax_e2e_qwen,
    {
        'mean': qwen3_latency_mean,
        'p95': qwen3_latency_p95,
        'p99': qwen3_latency_p99,
    },
 )
ax_e2e_qwen.set_title('E2E (Thinking)')
ax_e2e_qwen.set_yscale('log')

fig_e2e.supylabel('Query Count', x=0.04)
# fig_e2e.supxlabel('Latency (s)')
legend_handles = [
    Line2D([0], [0], **metric_styles['mean'], label='Mean'),
    Line2D([0], [0], **metric_styles['p95'], label='P95'),
    Line2D([0], [0], **metric_styles['p99'], label='P99'),
]
fig_e2e.legend(
    handles=legend_handles,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.1),
    ncol=3,
    frameon=True,
 )
sns.despine(fig=fig_e2e)
fig_e2e.tight_layout()
save_figure(fig_e2e, "figures/latency_distribution_e2e", formats=["pdf"])

# Figure 2: Latency per token distributions (All vs Thinking)
fig_tpt, axes_tpt = create_figure(
    "double_column",
    nrows=1,
    ncols=2,
    figsize=(9, 3.5),
)
axes_tpt = np.atleast_1d(axes_tpt).ravel()
ax_tpt_all, ax_tpt_qwen = axes_tpt

bins_latency_per_token = np.linspace(0, 0.5, 120)
ax_tpt_all.hist(
    latency_per_token_series,
    bins=bins_latency_per_token,
    color='#4C9A2A',
    edgecolor='white',
    alpha=0.8,
    label=None,
 )
add_metric_lines(
    ax_tpt_all,
    {
        'mean': latency_per_token_mean,
        'p95': latency_per_token_p95,
        'p99': latency_per_token_p99,
    },
 )
ax_tpt_all.set_title('TPT (All)')
ax_tpt_all.set_yscale('log')
ax_tpt_all.grid(linestyle=":", linewidth=2)
if not qwen3_latency_per_token_series.empty:
    ax_tpt_qwen.hist(
        qwen3_latency_per_token_series,
        bins=bins_latency_per_token,
        color='#F09553',
        edgecolor='white',
        alpha=0.8,
        label=None,
     )
    add_metric_lines(
        ax_tpt_qwen,
        {
            'mean': qwen3_latency_per_token_mean,
            'p95': qwen3_latency_per_token_p95,
            'p99': qwen3_latency_per_token_p99,
        },
     )
else:
    ax_tpt_qwen.text(
        0.5,
        0.5,
        'No Qwen3 samples with latency per token ≤0.5s',
        transform=ax_tpt_qwen.transAxes,
        ha='center',
        va='center',
        fontsize=10,
    )
ax_tpt_qwen.set_title('TPT (Thinking)')
ax_tpt_qwen.set_yscale('log')
ax_tpt_qwen.grid(linestyle=":", linewidth=2)
fig_tpt.supylabel('Query Count', x=0.04)
# fig_tpt.supxlabel('Latency per Token (s/token)')
sns.despine(fig=fig_tpt)
fig_tpt.tight_layout()
save_figure(fig_tpt, "figures/latency_distribution_tpt", formats=["pdf"])

/tmp/ipykernel_3940259/4101739883.py:135: UserWarning: The figure layout has changed to tight
  fig_e2e.tight_layout()
<frozen importlib._bootstrap>:491: RuntimeWarning: The global interpreter lock (GIL) has been enabled to load module 'fontTools.misc.bezierTools', which has not declared that it can run safely without the GIL. To override this behavior and keep the GIL disabled (at your own risk), run with PYTHON_GIL=0 or -Xgil=0.


Saved figure: figures/latency_distribution_e2e.pdf


/tmp/ipykernel_3940259/4101739883.py:201: UserWarning: The figure layout has changed to tight
  fig_tpt.tight_layout()


Saved figure: figures/latency_distribution_tpt.pdf


<Figure size 2700x1050 with 2 Axes>

<Figure size 2700x1050 with 2 Axes>

In [11]:
# Analyze total token usage per model with multi-tier piecewise power-law fit
max_models_bar = 20
token_source = df.dropna(subset=["total_tokens", "model"]).copy()

# Aggregate token usage by model
token_by_model = (
    token_source.groupby("model", as_index=False)["total_tokens"]
    .sum()
    .sort_values("total_tokens", ascending=False)
 )

# Take top N models
top_models_bar = token_by_model.head(max_models_bar).copy()
if len(top_models_bar) < 10:
    raise ValueError("Need at least 10 models with token data for comparison plots.")

# Remove provider prefixes (e.g., provider/model -> model)
top_models_bar["model_clean"] = top_models_bar["model"].str.replace(r"^.*/", "", regex=True)
top_models_bar["model_display"] = unique_truncated_labels(
    top_models_bar["model_clean"].tolist(),
    max_len=15,
 )
top_models_bar["model_rank"] = np.arange(1, len(top_models_bar) + 1)



rank = np.arange(1, len(top_models_bar) + 1, dtype=float)
tokens = top_models_bar["total_tokens"].to_numpy()
fit_result = fit_piecewise_zipf(rank, tokens, min_points=2, max_segments=3)

top_models_bar["pred_tokens"] = fit_result["pred"]
top_models_bar["tier_id"] = fit_result["tier_ids"]

num_tiers = max(1, len(fit_result["params"]))
tier_breakpoints = fit_result["breakpoints"]
tier_colors = pu.apply_color_palette("okabe_ito", max(2, num_tiers))[:num_tiers]
tier_markers = ["o", "s", "^", "D", "P", "h"]
tier_labels = [f"Tier {idx + 1} (α={params[0]:.2f})" for idx, params in enumerate(fit_result["params"])]

# Split into top-10 and ranks 11-20 for side-by-side visualization
top_10_models = top_models_bar.iloc[:10].copy()
next_10_models = top_models_bar.iloc[10:20].copy()

if next_10_models.empty:
    print("Warning: fewer than 20 models with token data; the right subplot will show the remaining models.")

# Create figure optimized for double column width
fig, (ax_left, ax_right) = create_figure(
    ncols=2,
    figsize=(10, 3.75),
    dpi=300,
    constrained_layout=False,
 )

# Get colors from the default palette for bar fills
colors = pu.apply_color_palette("default", len(top_models_bar))


def add_piecewise_curves(ax_obj, subset_df, y_positions, show_legend=True):
    """Overlay the fitted tier curves for the subset currently shown."""

    if subset_df.empty:
        return

    pos_array = np.asarray(y_positions)
    for tier_idx in range(num_tiers):
        tier_mask = subset_df["tier_id"].to_numpy() == tier_idx
        if not tier_mask.any():
            continue
        ax_obj.plot(
            subset_df.loc[tier_mask, "pred_tokens"],
            pos_array[tier_mask],
            color=tier_colors[tier_idx],
            linewidth=3,
            linestyle="--",
            marker=tier_markers[tier_idx % len(tier_markers)],
            markersize=6,
            markerfacecolor="white",
            markeredgewidth=2,
            label=tier_labels[tier_idx] if show_legend else None,
            zorder=10,
        )


def draw_regime_lines(ax_obj, subset_df, breakpoints, show_label=True):
    """Mark regime changes that fall within the subset's rank span."""

    if subset_df.empty:
        return

    start_rank = int(subset_df["model_rank"].min())
    end_rank = int(subset_df["model_rank"].max())
    label = "Regime Change" if show_label else None
    for bp in breakpoints:
        if start_rank <= bp <= end_rank:
            y_break = (bp - start_rank) + 0.5
            ax_obj.axhline(
                y=y_break,
                color="red",
                linestyle=":",
                linewidth=2.0,
                alpha=0.8,
                label=f"{label} (rank {bp})" if label else None,
                zorder=5,
            )
            label = None  # ensure label appears only once per axis

# Left subplot: Top 10 models
y_left = np.arange(len(top_10_models))
ax_left.barh(
    y_left,
    top_10_models["total_tokens"],
    color=colors[: len(top_10_models)],
    edgecolor="black",
    linewidth=0.5,
    alpha=0.7,
    label="Actual Usage",
 )
ax_left.set_yticks(y_left)
ax_left.set_yticklabels(top_10_models["model_display"])
ax_left.invert_yaxis()
add_piecewise_curves(ax_left, top_10_models, y_left, show_legend=True)
draw_regime_lines(ax_left, top_10_models, tier_breakpoints, show_label=True)
pu.style_axes(
    ax_left,
    title="Top 10 Models",
    xlabel="Total Tokens",
    ylabel="Model",
    grid=True,
    legend=True,
    legend_kwargs={"loc": "lower right", "fontsize": 9, "ncol": 2},
 )
ax_left.set_xscale("log")
ax_left.xaxis.set_major_formatter(FuncFormatter(human_readable_format))
# Right subplot: ranks 11-20 (compact labels)
if next_10_models.empty:
    ax_right.axis("off")
    ax_right.text(
        0.5,
        0.5,
        "Insufficient models for ranks 11-20",
        transform=ax_right.transAxes,
        ha="center",
        va="center",
        fontsize=11,
    )
else:
    y_right = np.arange(len(next_10_models))
    ax_right.barh(
        y_right,
        next_10_models["total_tokens"],
        color=colors[len(top_10_models) : len(top_10_models) + len(next_10_models)],
        edgecolor="black",
        linewidth=0.5,
        alpha=0.7,
    )
    ax_right.set_yticks(y_right)
    ax_right.set_yticklabels([f"#{int(rank)}" for rank in next_10_models["model_rank"]])
    ax_right.invert_yaxis()
    # put the axis label on the right side
    ax_right.yaxis.set_label_position("right")
    add_piecewise_curves(ax_right, next_10_models, y_right, show_legend=True)
    draw_regime_lines(ax_right, next_10_models, tier_breakpoints, show_label=False)

    pu.style_axes(
        ax_right,
        title="Top 11-20 Models",
        xlabel="Total Tokens",
        ylabel="Model Rank (#)",
        grid=True,
        legend=True,
    )
    ax_right.xaxis.set_major_formatter(FuncFormatter(human_readable_format))
    ax_right.xaxis.set_minor_formatter(NullFormatter())
    # remove offset like "×10^7"
    ax_right.xaxis.get_offset_text().set_visible(False)
# Adjust layout
handles, labels = ax_left.get_legend_handles_labels()
right_handles, right_labels = ax_right.get_legend_handles_labels()
# merge legends
handles = handles[:2] + [right_handles[0]] + handles[2:]
labels = labels[:2] + [right_labels[0]] + labels[2:]
fig.legend(handles=handles, labels=labels, ncols=3, bbox_to_anchor=(0.05, 1.25), loc=2, columnspacing=1.5, handletextpad=0.5)

# remove individual legends
ax_left.get_legend().remove()
ax_right.get_legend().remove()
fig.subplots_adjust(bottom=0.10, top=0.95, left=0.15, right=0.98, wspace=0.35)
fig.tight_layout()
pu.save_figure(
    fig,
    "figures/model_token_usage_bar_chart",
    formats=["pdf"],
    dpi=300,
 )

/tmp/ipykernel_3940259/2473327560.py:95: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(zipf_func, segment_rank, segment_tokens, p0=[1.0, segment_tokens[0]])


Saved figure: figures/model_token_usage_bar_chart.pdf


<Figure size 3000x1125 with 2 Axes>

In [12]:
# Single-figure view: Top 20 models without subplots
fig_top20, ax_top20 = create_figure(
    "single_column",
    nrows=1,
    ncols=1,
    figsize=(9, 5.75),
)

full_y = np.arange(len(top_models_bar))
colors_top20 = pu.apply_color_palette("default", len(top_models_bar))
ax_top20.barh(
    full_y,
    top_models_bar["total_tokens"],
    color=colors_top20,
    edgecolor="black",
    linewidth=0.5,
    alpha=0.8,
    label="Actual Usage",
)
ax_top20.tick_params(axis="y", labelsize=12)
ax_top20.set_yticks(full_y)
ax_top20.set_yticklabels(top_models_bar["model_display"])
ax_top20.invert_yaxis()
add_piecewise_curves(ax_top20, top_models_bar, full_y, show_legend=False)
draw_regime_lines(ax_top20, top_models_bar, tier_breakpoints, show_label=False)

# Label each bar with human-readable totals just beyond the bar tip
label_pad = 1.15
for ypos, value in zip(full_y, top_models_bar["total_tokens"]):
    if value <= 0:
        continue
    label_text = human_readable_format(value, None)
    ax_top20.text(
        value * label_pad,
        ypos,
        label_text,
        va="center",
        ha="left",
        fontsize=11,
        color="#333333",
        zorder=20,
    )

pu.style_axes(
    ax_top20,
    title="Top 20 Models by Token Usage",
    xlabel="Total Tokens",
    ylabel="Model",
    grid=True,
    legend=False,
)
ax_top20.set_xscale("log")
ax_top20.xaxis.set_major_formatter(FuncFormatter(human_readable_format))
ax_top20.xaxis.set_minor_formatter(NullFormatter())
ax_top20.xaxis.get_offset_text().set_visible(False)

legend_handles = [
    Line2D([0], [0], color="#666666", linewidth=6, alpha=0.6, label="Actual Usage"),
    *[
        Line2D(
            [0],
            [0],
            color=tier_colors[idx],
            linewidth=3,
            linestyle="--",
            marker=tier_markers[idx % len(tier_markers)],
            markersize=6,
            markerfacecolor="white",
            markeredgewidth=2,
            label=tier_labels[idx],
        )
        for idx in range(num_tiers)
    ],
    Line2D(
        [0],
        [0],
        color="red",
        linestyle=":",
        linewidth=2.0,
        label="Regime Change",
    ),
]
ax_top20.legend(
    handles=legend_handles,
    loc="lower right",
    frameon=True,
    fontsize=11,
)

sns.despine(fig_top20)
fig_top20.tight_layout()
save_figure(
    fig_top20,
    "figures/model_token_usage_top20_single",
    formats=["pdf"],
    dpi=300,
)

/tmp/ipykernel_3940259/635431041.py:91: UserWarning: The figure layout has changed to tight
  fig_top20.tight_layout()


Saved figure: figures/model_token_usage_top20_single.pdf


<Figure size 2700x1725 with 1 Axes>

In [13]:
minute_freq = "1min"
active_token_threshold = 1  # tokens required to treat a minute as "active"
model_minute_tokens = (
    df.set_index('created_at')
    .groupby('model')
    .resample(minute_freq)['total_tokens']
    .sum()
    .rename('tokens_per_min')
    .reset_index()
 )

def fill_missing_minutes(group: pd.DataFrame) -> pd.DataFrame:
    """Ensure each model covers every minute between its first and last observation."""
    if group.empty:
        return group
    full_index = pd.date_range(
        start=group['created_at'].min().floor(minute_freq),
        end=group['created_at'].max().ceil(minute_freq),
        freq=minute_freq,
    )
    filled = (
        group.set_index('created_at')
        .reindex(full_index, fill_value=0)
        .rename_axis('created_at')
        .reset_index()
    )
    filled['model'] = group.name
    return filled

model_minute_tokens = (
    model_minute_tokens.groupby('model', group_keys=False)
    .apply(fill_missing_minutes)
    .reset_index(drop=True)
 )

totals_per_model = (
    model_minute_tokens.groupby('model')['tokens_per_min']
    .agg(total_tokens='sum', total_minutes='size')
    .reset_index()
 )
model_minute_tokens = model_minute_tokens.merge(totals_per_model, on='model', how='left')

model_minute_tokens = model_minute_tokens[model_minute_tokens['total_tokens'] > 0].copy()
if model_minute_tokens.empty:
    raise ValueError("No models with positive total token counts after resampling.")

active_minutes = model_minute_tokens[
    model_minute_tokens['tokens_per_min'] >= active_token_threshold
].copy()
if active_minutes.empty:
    raise ValueError("No active minute windows found with the current threshold. Increase activity or lower the threshold.")

rpm_stats = (
    active_minutes.groupby('model')['tokens_per_min']
    .agg(
        mean_rpm='mean',
        peak_rpm='max',
        total_tokens_active='sum',
        active_minute_count='size',
    )
    .reset_index()
 )
rpm_stats['peak_to_mean_rpm'] = (
    rpm_stats['peak_rpm'] / rpm_stats['mean_rpm'].replace(0, np.nan)
)
burstiness_summary = rpm_stats.merge(
    totals_per_model.rename(columns={'total_tokens': 'total_tokens_observed', 'total_minutes': 'total_minutes_observed'}),
    on='model',
    how='left',
 )
burstiness_summary = (
    burstiness_summary
    .sort_values('peak_to_mean_rpm', ascending=False)
    .reset_index(drop=True)
 )

top_bursty_models = burstiness_summary.head(15).copy()
if top_bursty_models.empty:
    raise ValueError("Burstiness summary is empty; ensure the previous cell ran successfully.")

burstiness_plot_df = top_bursty_models.copy()
burstiness_plot_df['model_label'] = burstiness_plot_df['model'].str.replace(r'^.*/', '', regex=True)
burstiness_plot_df['model_label'] = unique_truncated_labels(
    burstiness_plot_df['model_label'].tolist(),
    max_len=14,
 )
colors = pu.apply_color_palette("default", len(burstiness_plot_df))
print(burstiness_plot_df)
fig_burst, ax_burst = create_figure(
    "single_column",
    nrows=1,
    ncols=1,
    figsize=(9, 5.75),
 )
ax_burst.barh(
    burstiness_plot_df['model_label'],
    burstiness_plot_df['peak_to_mean_rpm'],
    color=colors,
    edgecolor='black',
    linewidth=0.5,
    alpha=0.85,
 )
ax_burst.invert_yaxis()
ax_burst.grid(axis='x', linestyle=':', linewidth=1.2)
ax_burst.set_xlabel('Peak / Mean tokens-per-minute')
ax_burst.set_ylabel('Model')
ax_burst.set_title('Burstiness by Model (Peak vs Mean TPM)')
label_offset = max(burstiness_plot_df['peak_to_mean_rpm'].max() * 0.01, 0.05)
for model_label, value in zip(burstiness_plot_df['model_label'], burstiness_plot_df['peak_to_mean_rpm']):
    ax_burst.text(
        value + label_offset,
        model_label,
        f"{value:.2f}",
        va='center',
        ha='left',
        fontsize=11,
        color='#333333',
    )

sns.despine()
fig_burst.tight_layout()
save_figure(fig_burst, "figures/model_burstiness_peak_mean", formats=["pdf"])

                                        model       mean_rpm  peak_rpm  \
0           meta-llama/Llama-3.3-70B-Instruct  116986.350145  10286742   
1                              Qwen/Qwen3-32B  595520.978187  14306516   
2                swissai/Apertus-70B-Instruct    8262.244105    174263   
3                 apertus-70b-aligned-branded   10716.276853    207599   
4                 swissai/Apertus-8B-Instruct    3078.985801     58389   
5                          Apertus-8B-aligned    1780.229730     33353   
6                               Qwen/Qwen3-8B    5843.016791    101263   
7         Apertus-8B-aligned-branding-ckpt-18    5731.600000     98384   
8                swissai/apertus3-70b-10T-sft   12098.016492    201021   
9          Apertus-8B-aligned-branding-ckpt-9    1972.263158     32766   
10           Qwen/Qwen3-Next-80B-A3B-Thinking  132731.280294   1948382   
11                swissai/apertus3-70b-5T-sft    6047.548291     87744   
12        meta-llama/Meta-Llama-3-8B-I

/tmp/ipykernel_3940259/2761507325.py:121: UserWarning: The figure layout has changed to tight
  fig_burst.tight_layout()


Saved figure: figures/model_burstiness_peak_mean.pdf


<Figure size 2700x1725 with 1 Axes>

In [14]:
# Lifecycle-normalized burstiness trajectories across models
max_models_line = 8
num_lifecycle_bins = 80
min_duration_seconds = 60  # drop models observed for <1 minute to avoid divide-by-zero
selected_models = burstiness_summary['model'].head(max_models_line).tolist()
if not selected_models:
    raise ValueError("No models available for lifecycle burstiness plotting. Ensure previous cells ran successfully.")

lifecycle_bounds = (
    model_minute_tokens.groupby('model')['created_at']
    .agg(lifecycle_start='min', lifecycle_end='max')
    .reset_index()
)
lifecycle_bounds['lifecycle_duration_s'] = (
    lifecycle_bounds['lifecycle_end'] - lifecycle_bounds['lifecycle_start']
).dt.total_seconds()
valid_bounds = lifecycle_bounds[lifecycle_bounds['lifecycle_duration_s'] >= min_duration_seconds].copy()
if valid_bounds.empty:
    raise ValueError("All models have <1 minute of observations; cannot normalize lifecycle timelines.")

active_minute_stats = (
    active_minutes.groupby('model')['tokens_per_min']
    .agg(active_tokens='sum', active_minutes='size')
    .reset_index()
)
normalized_tokens = (
    active_minutes
    .merge(valid_bounds, on='model', how='inner')
    .merge(active_minute_stats, on='model', how='inner')
)
normalized_tokens['mean_tokens_per_active_min'] = (
    normalized_tokens['active_tokens']
    / normalized_tokens['active_minutes'].replace(0, np.nan)
)
normalized_tokens = normalized_tokens.replace([np.inf, -np.inf], np.nan)
normalized_tokens = normalized_tokens.dropna(subset=['mean_tokens_per_active_min'])
if normalized_tokens.empty:
    raise ValueError("Mean tokens per active minute unavailable; ensure active minute stats exist for selected models.")

normalized_tokens['burstiness_ratio'] = (
    normalized_tokens['tokens_per_min'] / normalized_tokens['mean_tokens_per_active_min']
)
print(normalized_tokens.head())
normalized_tokens = normalized_tokens.replace([np.inf, -np.inf], np.nan).dropna(subset=['burstiness_ratio'])
if normalized_tokens.empty:
    raise ValueError("Burstiness ratios could not be computed (possible zero means).")

normalized_tokens['lifecycle_position'] = (
    (normalized_tokens['created_at'] - normalized_tokens['lifecycle_start']).dt.total_seconds()
    / normalized_tokens['lifecycle_duration_s']
)
normalized_tokens['lifecycle_position'] = normalized_tokens['lifecycle_position'].clip(0, 1)
normalized_tokens['lifecycle_bin'] = (
    normalized_tokens['lifecycle_position'] * (num_lifecycle_bins - 1)
).round().astype(int)

plot_ready = (
    normalized_tokens
    [normalized_tokens['model'].isin(selected_models)]
    .groupby(['model', 'lifecycle_bin'], as_index=False)
    .agg(
        burstiness_mean=('burstiness_ratio', 'mean'),
        lifecycle_position=('lifecycle_position', 'mean'),
    )
)
if plot_ready.empty:
    raise ValueError("No lifecycle-normalized burstiness data available for the selected models.")

plot_ready = plot_ready.sort_values(['model', 'lifecycle_position']).reset_index(drop=True)
plot_ready['model_short'] = plot_ready['model'].str.replace(r'^.*/', '', regex=True)
model_label_lookup = plot_ready.groupby('model')['model_short'].first().to_dict()
unique_labels = unique_truncated_labels(list(model_label_lookup.values()), max_len=14)
model_label_lookup = dict(zip(model_label_lookup.keys(), unique_labels))
plot_ready['model_label'] = plot_ready['model'].map(model_label_lookup)

model_sequence = plot_ready['model'].unique().tolist()
colors = pu.apply_color_palette("okabe_ito", len(model_sequence))
fig_lifecycle, ax_lifecycle = create_figure(
    "single_column",
    nrows=1,
    ncols=1,
    figsize=(9, 3.75),
)
for idx, (model_name, model_data) in enumerate(plot_ready.groupby('model')):
    color = colors[idx % len(colors)]
    label = model_label_lookup[model_name]
    ax_lifecycle.plot(
        model_data['lifecycle_position'],
        model_data['burstiness_mean'],
        color=color,
        linewidth=2.0,
        alpha=0.85,
        label=label,
    )
ax_lifecycle.set_xlabel('Model Lifecycle')
ax_lifecycle.set_ylabel('Burstiness')
ax_lifecycle.set_title('Burstiness over Model Lifecycle')
ax_lifecycle.set_ylim(bottom=0)
ax_lifecycle.grid(linestyle=':', linewidth=1.0, alpha=0.9)
ax_lifecycle.legend(
    loc='upper right',
    bbox_to_anchor=(1.02, 1.02),
    frameon=True,
    ncol=2,
    fontsize=11,
)
sns.despine(fig_lifecycle)
fig_lifecycle.tight_layout()
save_figure(
    fig_lifecycle,
    "figures/model_burstiness_lifecycle",
    formats=["pdf"],
)

                 created_at  tokens_per_min                model  \
0 2025-08-29 14:57:00+00:00             742  Apertus-70B-aligned   
1 2025-08-29 14:58:00+00:00            2348  Apertus-70B-aligned   
2 2025-08-29 14:59:00+00:00            4288  Apertus-70B-aligned   
3 2025-08-29 15:00:00+00:00            1309  Apertus-70B-aligned   
4 2025-08-29 15:01:00+00:00            2101  Apertus-70B-aligned   

   total_tokens  total_minutes           lifecycle_start  \
0        100454           2537 2025-08-29 14:57:00+00:00   
1        100454           2537 2025-08-29 14:57:00+00:00   
2        100454           2537 2025-08-29 14:57:00+00:00   
3        100454           2537 2025-08-29 14:57:00+00:00   
4        100454           2537 2025-08-29 14:57:00+00:00   

              lifecycle_end  lifecycle_duration_s  active_tokens  \
0 2025-08-31 09:13:00+00:00              152160.0         100462   
1 2025-08-31 09:13:00+00:00              152160.0         100462   
2 2025-08-31 09:13:00+00:0

/tmp/ipykernel_3940259/2366544745.py:108: UserWarning: The figure layout has changed to tight
  fig_lifecycle.tight_layout()


Saved figure: figures/model_burstiness_lifecycle.pdf


<Figure size 2700x1125 with 1 Axes>

In [15]:
# Lifecycle-aligned burstiness trajectories across models (absolute time axis)
max_models_line = 8
min_duration_seconds = 60  # drop models observed for <1 minute to avoid divide-by-zero
selected_models = burstiness_summary['model'].head(max_models_line).tolist()
if not selected_models:
    raise ValueError("No models available for lifecycle burstiness plotting. Ensure previous cells ran successfully.")

lifecycle_bounds = (
    model_minute_tokens.groupby('model')['created_at']
    .agg(lifecycle_start='min', lifecycle_end='max')
    .reset_index()
)
lifecycle_bounds['lifecycle_duration_s'] = (
    lifecycle_bounds['lifecycle_end'] - lifecycle_bounds['lifecycle_start']
).dt.total_seconds()
valid_bounds = lifecycle_bounds[lifecycle_bounds['lifecycle_duration_s'] >= min_duration_seconds].copy()
if valid_bounds.empty:
    raise ValueError("All models have <1 minute of observations; cannot normalize lifecycle timelines.")

active_minute_stats = (
    active_minutes.groupby('model')['tokens_per_min']
    .agg(active_tokens='sum', active_minutes='size')
    .reset_index()
)
normalized_tokens = (
    model_minute_tokens
    .merge(valid_bounds, on='model', how='inner')
    .merge(active_minute_stats, on='model', how='inner')
)
normalized_tokens['mean_tokens_per_active_min'] = (
    normalized_tokens['active_tokens']
    / normalized_tokens['active_minutes'].replace(0, np.nan)
)
normalized_tokens = normalized_tokens.replace([np.inf, -np.inf], np.nan)
normalized_tokens = normalized_tokens.dropna(subset=['mean_tokens_per_active_min'])
if normalized_tokens.empty:
    raise ValueError("Mean tokens per active minute unavailable; ensure active minute stats exist for selected models.")

normalized_tokens['burstiness_ratio'] = (
    normalized_tokens['tokens_per_min'] / normalized_tokens['mean_tokens_per_active_min']
)
normalized_tokens = normalized_tokens.replace([np.inf, -np.inf], np.nan).dropna(subset=['burstiness_ratio'])
if normalized_tokens.empty:
    raise ValueError("Burstiness ratios could not be computed (possible zero means).")

normalized_tokens['elapsed_seconds'] = (
    normalized_tokens['created_at'] - normalized_tokens['lifecycle_start']
).dt.total_seconds()
normalized_tokens = normalized_tokens[normalized_tokens['elapsed_seconds'] >= 0].copy()
normalized_tokens['elapsed_hours'] = (
    normalized_tokens['elapsed_seconds'] / 3600.0
).round(3)

plot_ready = (
    normalized_tokens[normalized_tokens['model'].isin(selected_models)]
    .groupby(['model', 'elapsed_hours'], as_index=False)
    .agg(
        burstiness_mean=('burstiness_ratio', 'mean'),
    )
)
if plot_ready.empty:
    raise ValueError("No lifecycle-aligned burstiness data available for the selected models.")

plot_ready = plot_ready.sort_values(['model', 'elapsed_hours']).reset_index(drop=True)
plot_ready['model_short'] = plot_ready['model'].str.replace(r'^.*/', '', regex=True)
model_label_lookup = plot_ready.groupby('model')['model_short'].first().to_dict()
unique_labels = unique_truncated_labels(list(model_label_lookup.values()), max_len=14)
model_label_lookup = dict(zip(model_label_lookup.keys(), unique_labels))
plot_ready['model_label'] = plot_ready['model'].map(model_label_lookup)

model_sequence = plot_ready['model'].unique().tolist()
colors = pu.apply_color_palette("okabe_ito", len(model_sequence))
fig_lifecycle, ax_lifecycle = create_figure(
    "single_column",
    nrows=1,
    ncols=1,
    figsize=(9, 3.75),
)
for idx, (model_name, model_data) in enumerate(plot_ready.groupby('model')):
    color = colors[idx % len(colors)]
    label = model_label_lookup[model_name]
    ax_lifecycle.plot(
        model_data['elapsed_hours'],
        model_data['burstiness_mean'],
        color=color,
        linewidth=2.0,
        alpha=0.85,
        label=label,
    )
ax_lifecycle.set_xlabel('Hours Since First Request')
ax_lifecycle.set_ylabel('Burstiness')
ax_lifecycle.set_title('Burstiness over Time')
ax_lifecycle.set_ylim(bottom=0)
ax_lifecycle.set_xlim(left=0)
# y-axis to be log scale
# ax_lifecycle.set_yscale('log')
ax_lifecycle.grid(linestyle=':', linewidth=1.0, alpha=0.9)
ax_lifecycle.legend(
    loc='upper right',
    bbox_to_anchor=(1.02, 1.02),
    frameon=True,
    ncol=2,
    fontsize=11,
)
sns.despine(fig_lifecycle)
fig_lifecycle.tight_layout()
save_figure(
    fig_lifecycle,
    "figures/model_burstiness_absolute_time",
    formats=["pdf"],
)

/tmp/ipykernel_3940259/1720236053.py:106: UserWarning: The figure layout has changed to tight
  fig_lifecycle.tight_layout()


Saved figure: figures/model_burstiness_absolute_time.pdf


<Figure size 2700x1125 with 1 Axes>

In [16]:
max_models_line = 8
min_duration_seconds = 60
y_cap = 50

selected_models = burstiness_summary['model'].head(max_models_line).tolist()
if not selected_models:
    raise ValueError("No models available. Ensure previous cells ran successfully.")

# 1. Calculate Lifecycle Bounds
lifecycle_bounds = (
    model_minute_tokens.groupby('model')['created_at']
    .agg(lifecycle_start='min', lifecycle_end='max')
    .reset_index()
)
lifecycle_bounds['lifecycle_duration_s'] = (
    lifecycle_bounds['lifecycle_end'] - lifecycle_bounds['lifecycle_start']
).dt.total_seconds()
valid_bounds = lifecycle_bounds[lifecycle_bounds['lifecycle_duration_s'] >= min_duration_seconds].copy()

if valid_bounds.empty:
    raise ValueError("All models have <1 minute of observations.")

# 2. Calculate Active Mean (to be consistent with your Bar Chart)
active_minute_stats = (
    active_minutes.groupby('model')['tokens_per_min']
    .agg(active_tokens='sum', active_minutes='size')
    .reset_index()
)
normalized_tokens = (
    model_minute_tokens
    .merge(valid_bounds, on='model', how='inner')
    .merge(active_minute_stats, on='model', how='inner')
)

# Avoid divide by zero
normalized_tokens['mean_tokens_per_active_min'] = (
    normalized_tokens['active_tokens']
    / normalized_tokens['active_minutes'].replace(0, np.nan)
)
normalized_tokens = normalized_tokens.replace([np.inf, -np.inf], np.nan).dropna(subset=['mean_tokens_per_active_min'])

# 3. Calculate Burstiness Ratio
normalized_tokens['burstiness_ratio'] = (
    normalized_tokens['tokens_per_min'] / normalized_tokens['mean_tokens_per_active_min']
)
normalized_tokens = normalized_tokens.replace([np.inf, -np.inf], np.nan).dropna(subset=['burstiness_ratio'])

if normalized_tokens.empty:
    raise ValueError("Burstiness ratios could not be computed.")

# 4. Prepare Time Axis
normalized_tokens['elapsed_seconds'] = (
    normalized_tokens['created_at'] - normalized_tokens['lifecycle_start']
).dt.total_seconds()
normalized_tokens = normalized_tokens[normalized_tokens['elapsed_seconds'] >= 0].copy()
normalized_tokens['elapsed_hours'] = (
    normalized_tokens['elapsed_seconds'] / 3600.0
).round(3)

plot_ready = (
    normalized_tokens[normalized_tokens['model'].isin(selected_models)]
    .groupby(['model', 'elapsed_hours'], as_index=False)
    .agg(burstiness_mean=('burstiness_ratio', 'mean'))
)

if plot_ready.empty:
    raise ValueError("No data available for plotting.")

# 5. Clean Labels
plot_ready = plot_ready.sort_values(['model', 'elapsed_hours']).reset_index(drop=True)
plot_ready['model_short'] = plot_ready['model'].str.replace(r'^.*/', '', regex=True)
model_label_lookup = plot_ready.groupby('model')['model_short'].first().to_dict()
unique_labels = unique_truncated_labels(list(model_label_lookup.values()), max_len=14)
model_label_lookup = dict(zip(model_label_lookup.keys(), unique_labels))
plot_ready['model_label'] = plot_ready['model'].map(model_label_lookup)

# 6. Plotting
model_sequence = plot_ready['model'].unique().tolist()
colors = pu.apply_color_palette("okabe_ito", len(model_sequence))

fig_lifecycle, ax_lifecycle = create_figure(
    "single_column",
    nrows=1,
    ncols=1,
    figsize=(9, 3.75),
)

for idx, (model_name, model_data) in enumerate(plot_ready.groupby('model')):
    color = colors[idx % len(colors)]
    label = model_label_lookup[model_name]
    
    # Plot the line
    ax_lifecycle.plot(
        model_data['elapsed_hours'],
        model_data['burstiness_mean'],
        color=color,
        linewidth=2.0,
        alpha=0.85,
        label=label,
    )
    
    # --- NEW LOGIC: Check for peaks above the cap ---
    # Find the maximum burstiness point for this model
    max_point_idx = model_data['burstiness_mean'].idxmax()
    max_point = model_data.loc[max_point_idx]
    
    if max_point['burstiness_mean'] > y_cap:
        # Place text at the y_cap limit
        ax_lifecycle.text(
            max_point['elapsed_hours'], # X coordinate of the peak
            y_cap,                      # Y coordinate (clamped to the top)
            f" {max_point['burstiness_mean']:.2f}", # The text (e.g., " 88")
            color=color,
            fontsize=10,
            fontweight='bold',
            ha='center',    # Center horizontally on the peak
            va='bottom',    # Sit right on top of the border
            clip_on=False   # Allow text to render outside the plot area
        )

ax_lifecycle.set_xlabel('Hours Since First Request')
ax_lifecycle.set_ylabel('Burstiness')
ax_lifecycle.set_title('Burstiness over Time')
ax_lifecycle.set_xlim(left=0)

# Apply the Y-limit
ax_lifecycle.set_ylim(0, y_cap)

ax_lifecycle.grid(linestyle=':', linewidth=1.0, alpha=0.9)
ax_lifecycle.legend(
    loc='upper right',
    bbox_to_anchor=(1.02, 1.02),
    frameon=True,
    ncol=2,
    fontsize=11,
)

sns.despine(fig_lifecycle)
fig_lifecycle.tight_layout()
save_figure(
    fig_lifecycle,
    "figures/model_burstiness_absolute_time",
    formats=["pdf"],
)

/tmp/ipykernel_3940259/4161987396.py:139: UserWarning: The figure layout has changed to tight
  fig_lifecycle.tight_layout()


Saved figure: figures/model_burstiness_absolute_time.pdf


<Figure size 2700x1125 with 1 Axes>

In [19]:
qwen32b_df = df[df['model'] == 'Qwen/Qwen3-32B']
if qwen32b_df.empty:
    raise ValueError('No Qwen/Qwen3-32B records available for ratio analysis.')
qwen32b_ratio = qwen32b_df.sort_values('created_at')
token_denominator = qwen32b_ratio['total_tokens'].replace(0, np.nan)
qwen32b_ratio['output_ratio'] = qwen32b_ratio['reported_token_output'] / token_denominator
qwen32b_ratio = qwen32b_ratio.replace([np.inf, -np.inf], np.nan).dropna(subset=['output_ratio', 'created_at'])
if qwen32b_ratio.empty:
    raise ValueError('Output ratio series is empty after removing invalid rows.')
daily_request_counts = qwen32b_ratio['created_at'].dt.floor('D').value_counts().sort_index()
resample_freq = '1h'
zoom_freq = '1min'
hourly_ratio = (
    qwen32b_ratio.set_index('created_at')['output_ratio']
    .resample(resample_freq)
    .agg(['mean', 'var', 'count'])
    .rename(columns={'mean': 'ratio_mean', 'var': 'ratio_var', 'count': 'sample_count'})
    .reset_index()
)
hourly_ratio = hourly_ratio.dropna(subset=['ratio_mean'])
hourly_ratio['ratio_mean'] = hourly_ratio['ratio_mean'].clip(0, 1)
hourly_ratio['ratio_var'] = hourly_ratio['ratio_var'].fillna(0).clip(lower=0)
if hourly_ratio.empty:
    raise ValueError('No resampled buckets with valid ratio statistics to plot.')

std_band = np.sqrt(hourly_ratio['ratio_var'])
upper_band = np.clip(hourly_ratio['ratio_mean'] + std_band, 0, 1)
lower_band = np.clip(hourly_ratio['ratio_mean'] - std_band, 0, 1)

hourly_ratio['date'] = hourly_ratio['created_at'].dt.floor('D')
daily_counts = hourly_ratio.groupby('date')['sample_count'].sum().sort_values(ascending=False)
if daily_counts.empty:
    zoom_start = hourly_ratio['created_at'].max().floor('D') - pd.Timedelta(days=1)
else:
    zoom_start = daily_counts.index[0]
zoom_end = zoom_start + pd.Timedelta(days=1)

zoom_mask = (qwen32b_ratio['created_at'] >= zoom_start) & (qwen32b_ratio['created_at'] < zoom_end)
zoom_raw = qwen32b_ratio[zoom_mask].copy()
zoom_ratio = (
    zoom_raw.set_index('created_at')['output_ratio']
    .resample(zoom_freq)
    .agg(['mean', 'var', 'count'])
    .rename(columns={'mean': 'ratio_mean', 'var': 'ratio_var', 'count': 'sample_count'})
    .reset_index()
)
zoom_ratio = zoom_ratio.dropna(subset=['ratio_mean'])
if zoom_ratio.empty:
    raise ValueError('No 5-minute bins with data inside the zoom window.')
zoom_ratio['ratio_mean'] = zoom_ratio['ratio_mean'].clip(0, 1)
zoom_ratio['ratio_var'] = zoom_ratio['ratio_var'].fillna(0).clip(lower=0)

zoom_std = np.sqrt(zoom_ratio['ratio_var'])
zoom_upper = np.clip(zoom_ratio['ratio_mean'] + zoom_std, 0, 1)
zoom_lower = np.clip(zoom_ratio['ratio_mean'] - zoom_std, 0, 1)

fig_full, ax_full = create_figure(
    'single_column',
    nrows=1,
    ncols=1,
    figsize=(9, 3.75),
    sharex=False,
)
ax_full.plot(
    hourly_ratio['created_at'],
    hourly_ratio['ratio_mean'],
    color='#0B5F8A',
    linewidth=2.2,
    label='4h mean output share',
)
ax_full.fill_between(
    hourly_ratio['created_at'],
    lower_band,
    upper_band,
    color='#0B5F8A',
    alpha=0.15,
    label='±1 std dev',
)
ax_full.axvspan(zoom_start, zoom_end, color='#0B5F8A', alpha=0.08, label='Zoom window (busiest day)')
ax_full.set_ylabel('Output / Total tokens')
ax_full.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{y:.0%}'))
locator_full = mdates.AutoDateLocator(maxticks=5)
ax_full.xaxis.set_major_locator(locator_full)
ax_full.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator_full))
ax_full.grid(linestyle=':', linewidth=1.2, alpha=0.9)
sns.despine(fig_full)
fig_full.tight_layout()

fig_zoom, ax_zoom = create_figure(
    'single_column',
    nrows=1,
    ncols=1,
    figsize=(9, 3.75),
    sharex=False,
)
ax_zoom.plot(
    zoom_ratio['created_at'],
    zoom_ratio['ratio_mean'],
    color='#0B5F8A',
    linewidth=2.2,
    label='1 min mean output share',
)
ax_zoom.fill_between(
    zoom_ratio['created_at'],
    zoom_lower,
    zoom_upper,
    color='#0B5F8A',
    alpha=0.2,
    label='±1 std dev',
)
ax_zoom.set_ylabel('Output / Total tokens')
zoom_title = zoom_start.strftime('%Y-%m-%d') if isinstance(zoom_start, pd.Timestamp) else str(zoom_start)
ax_zoom.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{y:.0%}'))
locator_zoom = mdates.HourLocator(interval=6)
ax_zoom.xaxis.set_major_locator(locator_zoom)
ax_zoom.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
ax_zoom.grid(linestyle=':', linewidth=1.0, alpha=0.9)
ax_zoom.set_xlabel('Timestamp (UTC)')
sns.despine(fig_zoom)
fig_zoom.tight_layout()
save_figure(fig_full, 'figures/qwen32b_output_share_full', formats=['pdf'])
save_figure(fig_zoom, 'figures/qwen32b_output_share_zoom', formats=['pdf'])

/tmp/ipykernel_3940259/3557753238.py:87: UserWarning: The figure layout has changed to tight
  fig_full.tight_layout()
/tmp/ipykernel_3940259/3557753238.py:120: UserWarning: The figure layout has changed to tight
  fig_zoom.tight_layout()


Saved figure: figures/qwen32b_output_share_full.pdf
Saved figure: figures/qwen32b_output_share_zoom.pdf


<Figure size 2700x1125 with 1 Axes>

<Figure size 2700x1125 with 1 Axes>